# Single-rotor harmonic profile — recording vs reconstruction (DREGON singles)

What a stationary single-rotor recording is, measured on the DREGON single-motor recordings and
rebuilt from **three things only**: the mean shaft rate $\bar s$, the phase diffusion $D$, and the
harmonic amplitudes $A_k$ per microphone. No broadband floor, no fitting.

1. **Rotor strictly on**: the blade-pass line sits within 1 % of its plateau (0.25 s blocks, 0.5 s trimmed).
2. **$\bar s$**: median core-peak position of orders 2–7 in one Hann periodogram per mic (the cores are resolution-limited: the speed is constant to $1/T$). Known to ±0.003 rev/s.
3. **Jitter**: demodulate the BPF line, unwrap, detrend; the rms drift of the shaft phase from $\bar s t$ after 1 s is $r(1)$ ≈ 0.015–0.04 rev. It grows like $\sqrt{\tau}$ (a phase random walk, i.e. OU frequency noise with memory ≪ 0.1 s), so every harmonic is a **Lorentzian** of HWHM $\gamma_k = \pi k^2 D$ with $D = r(1)^2$.
4. **Amplitudes** (orders 1–150): the periodogram is averaged over $\pm\max(\gamma_k, 1/T)$ around $k\bar s$ and divided by the same average of the known line shape (Lorentzian through the window kernel) — the peak itself below the resolution, $\text{peak}\cdot\pi\gamma_k T/1.5$ once resolved. Everything under the line is the line.
5. **Reconstruction**: $x_c(t) = \sum_k A_{k,c}\cos(2\pi k\,\theta(t) + \phi_{k,c})$ with $\dot\theta = \bar s + $ white noise of diffusion $D$ (one shaft for all mics), random phases. Nothing else.

6. **Whittle fit** (`model` dropdown, second entry): the two-Lorentzian line $S_k = a_k[L_{\gamma_k} + \sigma_m^2 L_{\gamma_k+\gamma_m}] * K + F_k$ fitted to the same periodogram by the Whittle likelihood — amplitudes in closed form at every point, a grid + local search over $(D, \gamma_m, \sigma_m^2)$, the local floor $F_k$ a known constant (never a parameter), shape parameters from the non-windy mics (`results/static_rig/single_rotor/whittle/`, ~10–50 s each; `scripts/static_rig_scratch/wf_all.py`). $D$ comes out sharply; the spectral AM pair does not (profile flat below $\gamma_m \approx 0.1$ Hz).
7. **Amplitude jitter measured directly** (`amplitude_jitter`): 0.25 s Hann frames, peak of ±1 bin at $k\bar s$ per frame → $\ln A_k(t)$ per order and mic; its variance minus the additive-noise part is $\sigma_{m,k}^2$, its integral correlation time gives $\gamma_{m,k}$, and the cross-order correlation says whether the harmonics breathe together (`results/static_rig/single_rotor/amplitude_jitter.json`; `scripts/static_rig_scratch/am_all.py`). The second model's reconstruction uses the Whittle $D$ and these per-order $(\sigma_{m,k}^2, \gamma_{m,k})$ as independent log-amplitude OU processes per line.

Logic: `src/experiments/static_rig/single_rotor.py`; widgets: `notebooks/single_rotor_lab.py`.
The viewer is `plots.spectrum_viewer` with its new `overlay=`: both spectra on one pane (blue = recording,
red = reconstruction), the **spectrogram** toggle picks which one the top panel shows; all other
controls (overall / mean / frame, log axes, dB, band, dynamic range, click-to-pin) as before.
$\bar s$ and $r(1\,\mathrm{s})$ were measured once and are tabulated (`single_rotor_lab.TABLE`); nothing is re-estimated here. Windy mics per motor (`single_rotor_lab.WIND`) are kept out of the span detection; their profile is still read and rebuilt.

In [1]:
import sys

sys.path.insert(0, "../src")
sys.path.insert(0, ".")
import single_rotor_lab as L

In [2]:
lab = L.lab()
lab

## What to look at

- **Line heights** match within ~1 dB from k = 1 to the last order above the skirts (`overall (FFT)`, zoom with the band slider).
- **Line widths** grow as $k^2$ in both: compare the skirts of k = 20–40 (log amplitude, 60–80 dB range). Between the harmonics the reconstruction has only the skirts — the recording's broadband floor is deliberately absent, so the gap there is the floor.
- **Listening**: the reconstruction is the pure harmonic part; the recording adds the floor (and wind on the downwash mic).
- The **profile** (stems with caps) is the per-mic $20\log_{10} A_k$ of the selected model, for the channel selected in the viewer; its x-axis follows the viewer's band and axis type.
- **Windy mics** (Motor1: 5, 6; Motor2: 0; Motor3: 1, 2; Motor4: 4): their low-order amplitudes are wind, not rotor — read the other channels.
- **Whittle vs table**: compare the skirts at k = 10–40 (log amplitude, 60–80 dB range) and the pedestal around k = 1–4; the summary line gives the fitted D, γ_m, σ_m² and which grid D values are within 300 nats of the optimum.

In [ ]:
# every recording at a glance: given s̄ and D, detected span
L.precompute()